# Data extraction 2025

Cleans the 2025/2026 Qualtrics export into the analysis-ready CSVs. The association and
valence tasks are extracted with the shared helpers in `data_cleaning_qualtrics`; the
2025-specific participant-ID scheme and the questionnaire parsing stay in the notebook.

In [1]:
import pandas as pd

from config import RESSERENO_ROOT
from data_cleaning_qualtrics import (
    get_associations,
    get_valences,
    associations_valences_df,
)

pd.set_option("display.max_rows", None)

path_data = RESSERENO_ROOT / "data"
path_out = RESSERENO_ROOT / "out" / "formatting" / "all_students_2025"

## Data extraction

In [2]:
# data_spring_2026 already re-includes every data_2026 response, so use it instead of data_2026
# (adding both would double-count the 2026 batch). its two Qualtrics header rows are dropped here;
# the 2025 header rows are removed together during trimming below.
raw_2025 = pd.read_csv(f"{path_data}/data_2025.csv")
raw_spring_2026 = pd.read_csv(f"{path_data}/data_2026.csv")[2:]
raw_data = pd.concat([raw_2025, raw_spring_2026], ignore_index=True)

In [3]:
DROP_COLUMNS = [
    "EndDate",
    "Status",
    "IPAddress",
    "RecordedDate",
    "ResponseId",
    "RecipientLastName",
    "RecipientFirstName",
    "RecipientEmail",
    "ExternalReference",
    "LocationLatitude",
    "LocationLongitude",
    "DistributionChannel",
    "UserLanguage",
]

# drop unused columns and the two leftover Qualtrics header rows
data = raw_data.drop(columns=DROP_COLUMNS)[2:]

# keep only the participations with >65% progress that gave consent (QID3 == "1")
progress = data["Progress"].values.astype(int)
data = data[(progress > 65) & (data["QID3"] == "1")].reset_index(drop=True)

## Participant ID creation

Participants come from several populations, told apart by the population question, the day they took
part (`StartDate`), or the session time slot.

Two independent survey revisions matter:
- **27 Feb 2026** — the cue word shown as *"Meditazione"* became *"ChatGPT"* (handled in the
  associations step via `CHATGPT_CUTOFF`).
- **4 May 2026** — the questionnaire was restructured: `Q20` became *studente/docente*, and the
  department question moved to `Q21` (asked to students only), so teachers have an empty `Q21`.
  Before this date the department was in `Q20`.

IDs are assigned per survey era, continuing the numbering of shared groups so the eras can be
analysed together.

In [4]:
# 2026-02-27: the cue word shown as "Meditazione" became "ChatGPT" (used in the associations step)
CHATGPT_CUTOFF = pd.Timestamp("2026-02-27")


def assign_id_by_question(df, q20_value, prefix, era_mask, start_idx=1, idx="Q20"):
    """Assign participant_id from the Q20 answer, within a single survey era."""
    mask = era_mask & (df[idx] == q20_value)
    df.loc[mask, "participant_id"] = [
        f"{prefix}_{i:03d}" for i in range(start_idx, start_idx + mask.sum())
    ]


def assign_id_by_date(df, dates, prefix, mask_extra=None):
    """Assign participant_id to everyone who took part on the given StartDate day(s)."""
    if isinstance(dates, str):
        dates = [dates]
    days = {pd.to_datetime(d).date() for d in dates}
    mask = df["StartDate"].dt.date.isin(days)
    if mask_extra is not None:
        mask &= mask_extra
    suffix = "_1" if prefix == "fisica_stem" else ""
    df.loc[mask, "participant_id"] = [
        f"{prefix}_{i:03d}{suffix}" for i in range(1, mask.sum() + 1)
    ]


def assign_id_by_time(df, start_time, end_time, prefix, start_idx=1):
    """Assign participant_id to liceo students (Q20 == '3') within a StartDate interval."""
    mask = (
        (df["StartDate"] >= start_time)
        & (df["StartDate"] < end_time)
        & (df["Q20"] == "3")
    )
    df.loc[mask, "participant_id"] = [
        f"{prefix}_{i:03d}" for i in range(start_idx, start_idx + mask.sum())
    ]
    return mask, start_idx + mask.sum()

In [5]:
data["StartDate"] = pd.to_datetime(data["StartDate"])

# researcher test run on 2026-05-04 ("testo finale STELLA PROVA"), not a participant
data = data[data["StartDate"] != pd.Timestamp("2026-05-04 15:05:21")].reset_index(drop=True)

# create the column up front with real missing values: letting the first .loc assignment create it
# fills the unassigned rows with the string "nan", which the teacher isna() check below can't see
data["participant_id"] = pd.Series(pd.NA, index=data.index, dtype="str")

# on 2026-05-04 the survey was restructured: Q20 became studente/docente and the department
# question moved to Q21 (students only). before that date the department was in Q20.
AFFILIATION_CUTOFF = pd.Timestamp("2026-05-04")
pre_affiliation = data["StartDate"] < AFFILIATION_CUTOFF
post_affiliation = ~pre_affiliation

# --- old survey (department in Q20): 1 fisica / 2 psicologia / 3 liceo ---
assign_id_by_question(data, "1", "fisica", pre_affiliation)
assign_id_by_question(data, "2", "psicologia", pre_affiliation)
assign_id_by_question(data, "3", "liceo", pre_affiliation)

# interfaces study (single day) and STEM-anxiety listening group (multiple sessions)
assign_id_by_date(data, "2025-03-26", "interfacce")
assign_id_by_date(
    data,
    ["2025-05-21", "2025-05-29", "2025-06-17", "2025-11-10", "2026-01-08"],
    "fisica_stem",
)

# split the liceo students into scientifico / linguistico by session time slot
mask_sci, idx_sci = assign_id_by_time(
    data, "2025-05-13 11:40:00", "2025-05-13 11:45:00", "liceo_sci", start_idx=1
)
assign_id_by_time(
    data, "2025-05-13 12:27:00", "2025-05-13 12:40:00", "liceo_lin", start_idx=1
)
# start the second scientifico session one number later, leaving a slot for the late
# participant below (row 65: a lone liceo scientifico student a few days later)
assign_id_by_time(
    data,
    "2025-05-30 07:50:00",
    "2025-05-30 08:55:00",
    "liceo_sci",
    start_idx=idx_sci + 1,
)
data.loc[65, "participant_id"] = f"liceo_sci_{mask_sci.sum() + 1:03d}"

# STEM group: participants who came back for later rounds of questionnaires
RETURNING_STEM = [
    ("2025-07-22", "fisica_stem_002_2"),
    ("2025-09-29", "fisica_stem_001_2"),
    ("2025-11-19", "fisica_stem_002_3"),
]
for date_str, pid in RETURNING_STEM:
    mask = data["StartDate"].dt.date == pd.to_datetime(date_str).date()
    assert (
        mask.sum() == 1
    ), f"expected exactly one row on {date_str}, found {mask.sum()}"
    data.loc[mask, "participant_id"] = pid

# --- new survey (department in Q21, students only): 1 fisica_unitn / 2 fisica / 3 psicologia / 4 liceo / 5 altro ---
# fisica_unitn is new; fisica/psicologia/liceo continue the numbering of their old-survey
# counterparts. Counts are taken from participant_id itself (not raw Q20) so they reflect
# what's actually left after the interfacce/fisica_stem/liceo_sci/liceo_lin overwrites above.
n_fisica_pre = data["participant_id"].str.fullmatch(r"fisica_\d+").sum()
n_psico_pre = data["participant_id"].str.fullmatch(r"psicologia_\d+").sum()
n_liceo_pre = data["participant_id"].str.fullmatch(r"liceo_\d+").sum()

assign_id_by_question(data, "1", "fisica_unitn", post_affiliation, idx="Q21")
assign_id_by_question(
    data, "2", "fisica", post_affiliation, start_idx=n_fisica_pre + 1, idx="Q21"
)
assign_id_by_question(
    data, "3", "psicologia", post_affiliation, start_idx=n_psico_pre + 1, idx="Q21"
)
assign_id_by_question(
    data, "4", "liceo", post_affiliation, start_idx=n_liceo_pre + 1, idx="Q21"
)
assign_id_by_question(
    data, "5", "altro dipartimento", post_affiliation, idx="Q21"
)  # new group, starts at 1

# teachers: new-survey participants who skipped the department question (Q21 is NaN).
# Guarded on participant_id still being empty, so an affiliation-cutoff mistake can't
# silently overwrite an ID assigned above.
is_teacher = post_affiliation & data["Q21"].isna() & data["participant_id"].isna()
data.loc[is_teacher, "participant_id"] = [
    f"teacher_{i:03d}" for i in range(1, is_teacher.sum() + 1)
]

assign_id_by_date(
    data,
    ["2026-09-07", "2026-09-08", "2026-09-09", "2026-09-10", "2026-09-11"],
    "fisica_lecce",
)

# keep only the first session of each returning STEM-group participant: later rounds are
# repeated measures of the same people
data = data[
    ~data["participant_id"].isin([pid for _, pid in RETURNING_STEM])
].reset_index(drop=True)

# --- sanity checks ---
n_missing = data["participant_id"].isna().sum()
n_dupes = data["participant_id"].duplicated().sum()
if n_missing:
    print(f"WARNING: {n_missing} rows have no participant_id")
if n_dupes:
    print(f"WARNING: {n_dupes} duplicate participant_id values")

print(f"Participants: {len(data)} ({n_missing} unassigned, {n_dupes} duplicates)")
print(
    data["participant_id"]
    .str.replace(r"_\d.*", "", regex=True)
    .value_counts()
    .to_string()
)

Participants: 193 (0 unassigned, 0 duplicates)
participant_id
fisica_lecce          44
liceo_sci             43
fisica                36
interfacce            23
liceo_lin             19
teacher                8
fisica_unitn           8
fisica_stem            5
altro dipartimento     4
liceo                  2
psicologia             1


## Associations & valence tasks

`get_associations` / `get_valences` do the heavy lifting; a few relabellings are applied afterwards.

In [6]:
associations_task_df = get_associations(data)

# the "Meditazione" slot was renamed "ChatGPT" from the revision date on (config labels it "ChatGPT");
# relabel everyone who took part before the cutoff back to "Meditazione"
chatgpt_ids = set(data.loc[data["StartDate"] >= CHATGPT_CUTOFF, "participant_id"])
relabel_meditazione = (
    associations_task_df["cue_word"] == "ChatGPT"
) & ~associations_task_df["participant_id"].isin(chatgpt_ids)
associations_task_df.loc[relabel_meditazione, "cue_word"] = "Meditazione"

# for the early 'interfacce' participants that slot was still "Counselling"
interfacce_ai = associations_task_df["participant_id"].astype(str).str.startswith(
    "interfacce"
) & (associations_task_df["cue_word"] == "Intelligenza_Artificiale")
associations_task_df.loc[interfacce_ai, "cue_word"] = "Counselling"

# lowercase all free text
associations_task_df = associations_task_df.map(
    lambda x: x.lower() if isinstance(x, str) else x
)

In [7]:
valence_task_df = get_valences(data)
associations_valence_df = associations_valences_df(
    associations_task_df, valence_task_df
)

### Missing-data filtering & export

In [8]:
# mean fraction of missing answers per participant
missing_fraction = (
    associations_valence_df.set_index("participant_id")
    .isna()
    .mean(axis=1)
    .groupby(level=0)
    .mean()
)

# drop participants missing more than a third of their data (same rule for every group)
to_drop = missing_fraction[missing_fraction > 1 / 3].index
print("Participants to drop:", len(to_drop), list(to_drop))

associations_valence = associations_valence_df[
    ~associations_valence_df["participant_id"].isin(to_drop)
].reset_index(drop=True)
associations_valence.to_csv(
    f"{path_out}/associations_valence_task_qualtrics.csv", index=False
)

Participants to drop: 30 ['altro dipartimento_002', 'fisica_001', 'fisica_002', 'fisica_004', 'fisica_006', 'fisica_008', 'fisica_009', 'fisica_019', 'fisica_023', 'fisica_025', 'fisica_lecce_001', 'fisica_lecce_002', 'fisica_lecce_003', 'fisica_lecce_007', 'fisica_lecce_010', 'fisica_lecce_013', 'fisica_lecce_014', 'fisica_lecce_019', 'fisica_lecce_023', 'fisica_lecce_025', 'fisica_lecce_026', 'fisica_lecce_031', 'fisica_lecce_036', 'fisica_lecce_044', 'fisica_stem_004_1', 'fisica_unitn_001', 'liceo_lin_019', 'liceo_sci_015', 'psicologia_001', 'teacher_008']


In [9]:
mask = associations_valence["participant_id"].str.startswith("fisica_lecce")
associations_valence.loc[mask, "participant_id"].nunique()

30

## Questionnaires

Every questionnaire block is extracted with a single helper, then filtered on `to_drop` and
exported.

In [10]:
# gender and math self-perception / grade
gender = (
    data["Q17"].replace({"1": "M", "2": "F", "4": "NB", "5": "NA"}).rename("gender")
)
percieved_abilities = data["Q16"].rename("percieved_abilities")

# math grade: prefer the university grade (Q20_1) over the liceo one (Q21_1)
math_grade_uni = data["Q20_1"].replace(31, "30L")
math_grade_liceo = data["Q21_1"].replace(11, "10L")
math_grade = math_grade_uni.combine_first(math_grade_liceo).rename("math_grade")

In [11]:
def extract_block(data, question_id, start, end, col_label):
    """Pull a questionnaire's item columns ('{question_id}_{i}') into a labelled block."""
    block = data[[f"{question_id}_{i}" for i in range(start, end)]].copy()
    block.columns = [f"{col_label}_{i}" for i in range(start, end)]
    return block


cfs_data = extract_block(data, "Q22#1", 1, 13, "cfs")  # Cognitive Flexibility Scale
stai_data = extract_block(data, "QID77", 1, 19, "state_anxiety")  # STAI: state anxiety
big5_data = extract_block(data, "QID84", 1, 11, "big_5")  # Big Five (10 items)

# MAS_IT (reduced): keep the original, non-contiguous item numbering
mas_it_data = extract_block(data, "Q21#1", 1, 19, "mas_it")
mas_it_items = [1, 3, 4, 5, 6, 7, 8, 9, 11, 12, 13, 14, 18, 19, 20, 21, 22, 23]
mas_it_data.columns = [f"mas_it_{n}" for n in mas_it_items]

In [12]:
def export(frame, filename):
    """Drop the excluded participants and write the frame to CSV."""
    frame = frame[~frame["participant_id"].isin(to_drop)].reset_index(drop=True)
    frame.to_csv(f"{path_out}/{filename}", index=False)
    return frame


mas_it_df = export(
    pd.concat([data["participant_id"], mas_it_data], axis=1), "mas_it_2025.csv"
)

all_questionnaires = export(
    pd.concat(
        [
            data["participant_id"],
            gender,
            percieved_abilities,
            math_grade,
            cfs_data,
            stai_data,
            big5_data,
            mas_it_data,
        ],
        axis=1,
    ),
    "all_questionnaires_2025.csv",
)

## Free text

In [13]:
# lowercase, strip trailing punctuation/whitespace, then end with a single full stop
text = (data["Q18"].str.lower().str.rstrip(".!? \n\t") + ".").rename("text")
text_df = export(pd.concat([data["participant_id"], text], axis=1), "free_text.csv")